

Simple Example of Data Skew + Salting in Spark

Let’s walk through a realistic, small-scale example of how data skew happens and how salting fixes it.

1. Simulate Skewed Data

Imagine you have two datasets:

Large dataset (transactions) — skewed by cust_id:

In [0]:
#Large dataset (transactions)
from pyspark.sql import SparkSession
from pyspark.sql.functions import rand

spark = SparkSession.builder.appName("SaltingExample").getOrCreate()

# Skewed customer data: one customer has many transactions
transactions = [
    ("C001", 100.0, "Laptop"),
    ("C001", 200.0, "Mouse"),
    ("C001", 150.0, "Keyboard"),
    ("C001", 300.0, "Monitor"),
    ("C001", 50.0, "Cable"),
    ("C002", 400.0, "Tablet"),
    ("C003", 250.0, "Phone")
]
df1 = spark.createDataFrame(transactions, ["cust_id", "amount", "product"])
print("Here, C001 has 5 transactions, while others have 1 each.")
display(df1)

In [0]:
#Small dataset (profiles):

profiles = [
    ("C001", "Alice"),
    ("C002", "Bob"),
    ("C003", "Charlie")
]
df2 = spark.createDataFrame(profiles, ["cust_id", "name"])

display(df2)

2. Problem Without Salting

If you join normally:

In [0]:
#All 5 transactions for C001 go to one partition, while others have 1 each. This creates data skew — one partition is overloaded, others idle Towards Dev.


result = df1.join(df2, "cust_id")

#### 3. Apply Salting

We’ll split the large dataset’s cust_id into multiple “salted” keys and expand the small dataset to match.

In [0]:
# Define number of salt buckets
num_salting_keys = 5

# Add salt to large dataset
from pyspark.sql.functions import col, lit, concat_ws

df1_salted = df1.withColumn("salt", (rand() * num_salting_keys).cast("int"))
df1_salted = df1_salted.withColumn("cust_id_salted", concat_ws("_", col("cust_id"), col("salt")))

# Expand small dataset to match all salted keys
from pyspark.sql.functions import explode

# Create all possible salted keys for each cust_id in df2
df2_expanded = df2.withColumn("salt", lit(0))  # placeholder
df2_expanded = df2_expanded.withColumn("cust_id_salted", concat_ws("_", col("cust_id"), col("salt")))
df2_expanded = df2_expanded.withColumn("cust_id", col("cust_id_salted").substr(0, 4))  # extract original cust_id

# Expand to all salt values
df2_expanded = df2_expanded.withColumn("cust_id_salted", concat_ws("_", col("cust_id"), col("salt")))
df2_expanded = df2_expanded.withColumn("cust_id", col("cust_id_salted").substr(0, 4))
df2_expanded = df2_expanded.withColumn("salt", lit(0))  # reset salt for join

# Join on salted keys
result_salted = df1_salted.join(df2_expanded, "cust_id_salted")

#### 4. How It Works

Salting: Append a random integer to the cust_id in the large dataset, splitting C001 into C001_0, C001_1, ..., C001_4 www.sparkplayground.com.

**Expansion:** The small dataset is expanded to include all salted keys so each partition has a balanced load.

Join: Spark processes each salted key in parallel, avoiding the bottleneck.

#### 5. Benefits
Balanced partitions: No single partition holds most of the data.

Faster joins: Workload is distributed evenly.

Better resource utilization: All executors work instead of idle ones practical-software.com.

**Tip:** Choose num_salting_keys based on your cluster’s number of executors — more keys = better distribution but more overhead.